# U.S. feature audit

This training-only audit compares the saved conservative HGB recipe with a small set of pre-specified launch-input ablations. It uses the official U.S. training IDs, the same three outer partitions as the current validation, and refits feature construction, target encoding, imputation, and scaling inside every fold. It never loads reserved IDs or labels.

The safe additions are tested one at a time: state-relative goal, round-$1,000 goal, exact 30-day duration, and exact 60-day duration. These are diagnostics only; no model artifact is changed.

In [1]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'preprocessing.py').is_file())
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from utils.us_modeling import load_us_training_data
from utils.preprocessing import FeatureBuilder, TargetEncoder

X, y, ids = load_us_training_data(PROJECT_ROOT)
print(f'Official U.S. training rows: {len(X):,}; reserved labels were not loaded')

Official U.S. training rows: 26,945; reserved labels were not loaded


## Configurations

The baseline retains the exact current HGB inputs. “No state/location” drops the state target encoding; “no duration-derived” removes `goal_per_day` but retains raw duration; “no category-relative” removes both category-relative goal features; “no redundant goal-derived” removes the relative goal features and retains `log_goal`/`goal_per_day`. The last four rows add one safe candidate at a time.

In [2]:
class AuditBuilder(FeatureBuilder):
    def __init__(self, drop=None, add=None):
        super().__init__(include_timing=False, include_flags=False, include_reward=False, exclude_renamed=True)
        self.drop = tuple(drop or ())
        self.add = tuple(add or ())

    def fit(self, X, y=None):
        super().fit(X, y)
        if 'goal_vs_state' in self.add:
            log_goal = np.log1p(pd.to_numeric(X['goal'], errors='coerce'))
            state = X['location'].astype('string').str.extract(r',\s*([A-Z]{2})\s*$', expand=False)
            tmp = pd.DataFrame({'state': state.to_numpy(), 'log_goal': log_goal.to_numpy()})
            self.state_median_ = tmp.groupby('state', dropna=True)['log_goal'].median()
            self.overall_state_median_ = float(log_goal.median())
        return self

    def transform(self, X):
        out = super().transform(X)
        log_goal = np.log1p(pd.to_numeric(X['goal'], errors='coerce'))
        duration = pd.to_numeric(X['duration'], errors='coerce')
        if 'goal_vs_state' in self.add:
            state = X['location'].astype('string').str.extract(r',\s*([A-Z]{2})\s*$', expand=False)
            out['goal_vs_state'] = (log_goal - state.map(self.state_median_).fillna(self.overall_state_median_)).to_numpy()
        if 'goal_round_1000' in self.add: out['goal_round_1000'] = X['goal'].mod(1000).eq(0).astype(float).to_numpy()
        if 'duration_exact_30' in self.add: out['duration_exact_30'] = duration.eq(30).astype(float).to_numpy()
        if 'duration_exact_60' in self.add: out['duration_exact_60'] = duration.eq(60).astype(float).to_numpy()
        return out.drop(columns=list(self.drop), errors='ignore')

BASE_CAT = ['category', 'subcategory', 'state']

def audit_pipeline(name):
    drop, add, cats = (), (), list(BASE_CAT)
    if name == 'without state/location': drop, cats = ('state',), ['category', 'subcategory']
    elif name == 'without duration-derived': drop = ('goal_per_day',)
    elif name == 'without category-relative goal': drop = ('goal_vs_cat', 'goal_vs_subcat')
    elif name == 'without redundant goal-derived': drop = ('goal_vs_cat', 'goal_vs_subcat')
    elif name.startswith('add '): add = (name[4:],)
    numeric = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
    prep = ColumnTransformer([('num', numeric, make_column_selector(dtype_include=np.number)), ('cat', TargetEncoder(cols=cats), cats)], remainder='drop', verbose_feature_names_out=False)
    model = HistGradientBoostingClassifier(l2_regularization=0.0026915045092098205, learning_rate=0.04318571864689941, max_iter=219, max_leaf_nodes=14, min_samples_leaf=43, random_state=42)
    return Pipeline([('features', AuditBuilder(drop=drop, add=add)), ('prep', prep), ('model', model)])

configs = ['original conservative', 'without state/location', 'without duration-derived', 'without category-relative goal', 'without redundant goal-derived', 'add goal_vs_state', 'add goal_round_1000', 'add duration_exact_30', 'add duration_exact_60']
# The two relative-goal removals answer different audit questions: the first isolates the family; the second treats it as redundant with log_goal and goal_per_day.
print(configs)

['original conservative', 'without state/location', 'without duration-derived', 'without category-relative goal', 'without redundant goal-derived', 'add goal_vs_state', 'add goal_round_1000', 'add duration_exact_30', 'add duration_exact_60']


In [3]:
outer = StratifiedKFold(n_splits=3, shuffle=True, random_state=20261027)
records = []
all_prob = {}
for config in configs:
    prob = np.empty(len(y))
    for fold, (fit_idx, valid_idx) in enumerate(outer.split(X, y), 1):
        pipe = audit_pipeline(config if config != 'original conservative' else 'original')
        started = time.perf_counter()
        pipe.fit(X.iloc[fit_idx], y.iloc[fit_idx])
        fit_seconds = time.perf_counter() - started
        p = pipe.predict_proba(X.iloc[valid_idx])[:, 1]
        prob[valid_idx] = p
        pred = p >= 0.5
        tn, fp, fn, tp = confusion_matrix(y.iloc[valid_idx], pred, labels=[0, 1]).ravel()
        records.append({'config': config, 'fold': fold, 'accuracy': accuracy_score(y.iloc[valid_idx], pred), 'roc_auc': roc_auc_score(y.iloc[valid_idx], p), 'success_f1': f1_score(y.iloc[valid_idx], pred, zero_division=0), 'failure_recall': recall_score(y.iloc[valid_idx], pred, pos_label=0, zero_division=0), 'fit_seconds': fit_seconds, 'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp})
    all_prob[config] = prob
fold_results = pd.DataFrame(records)
summary = fold_results.groupby('config', sort=False).agg(accuracy=('accuracy','mean'), accuracy_sd=('accuracy','std'), roc_auc=('roc_auc','mean'), roc_auc_sd=('roc_auc','std'), success_f1=('success_f1','mean'), failure_recall=('failure_recall','mean'), fit_seconds=('fit_seconds','mean')).reset_index()
summary.round(4)

,config,accuracy,accuracy_sd,roc_auc,roc_auc_sd,success_f1,failure_recall,fit_seconds
0,original conservative,0.6693,0.0005,0.7240,0.0005,0.7158,0.5572,1.0108
1,without state/location,0.6616,0.0020,0.7106,0.0018,0.7098,0.5458,0.8791
2,without duration-derived,0.6707,0.0021,0.7240,0.0005,0.7170,0.5587,0.9488
3,without category-relative goal,0.6712,0.0020,0.7224,0.0008,0.7167,0.5625,0.8487
4,without redundant goal-derived,0.6712,0.0020,0.7224,0.0008,0.7167,0.5625,0.7833
5,add goal_vs_state,0.6720,0.0013,0.7244,0.0008,0.7184,0.5588,0.8356
6,add goal_round_1000,0.6711,0.0009,0.7239,0.0004,0.7178,0.5569,0.9145
7,add duration_exact_30,0.6713,0.0026,0.7241,0.0003,0.7172,0.5606,1.7435
8,add duration_exact_60,0.6712,0.0020,0.7242,0.0004,0.7177,0.5581,1.3845


In [4]:
# Pooled metrics and fold consistency relative to the original.
def pooled(config):
    p = all_prob[config]; pred = p >= 0.5
    return {'config': config, 'accuracy': accuracy_score(y, pred), 'roc_auc': roc_auc_score(y, p), 'success_f1': f1_score(y, pred, zero_division=0), 'failure_recall': recall_score(y, pred, pos_label=0, zero_division=0)}
pooled_results = pd.DataFrame([pooled(c) for c in configs])
base = fold_results[fold_results.config.eq('original conservative')].set_index('fold')
consistency = []
for c in configs[1:]:
    cur = fold_results[fold_results.config.eq(c)].set_index('fold')
    consistency.append({'config': c, 'folds_accuracy_higher': int((cur.accuracy > base.accuracy).sum()), 'fold_accuracy_delta': (cur.accuracy - base.accuracy).round(4).tolist(), 'fold_auc_delta': (cur.roc_auc - base.roc_auc).round(4).tolist(), 'fold_failure_recall_delta': (cur.failure_recall - base.failure_recall).round(4).tolist()})
print('Pooled outer-fold metrics')
display(pooled_results.round(4))
print('Mean fold metrics, variability, and runtime')
display(summary.round(4))
print('Fold deltas versus original')
display(pd.DataFrame(consistency))

Pooled outer-fold metrics


,config,accuracy,roc_auc,success_f1,failure_recall
0,original conservative,0.6693,0.7239,0.7158,0.5572
1,without state/location,0.6616,0.7105,0.7098,0.5458
2,without duration-derived,0.6707,0.7239,0.7170,0.5587
3,without category-relative goal,0.6712,0.7223,0.7167,0.5625
4,without redundant goal-derived,0.6712,0.7223,0.7167,0.5625
5,add goal_vs_state,0.6720,0.7243,0.7184,0.5588
6,add goal_round_1000,0.6711,0.7238,0.7178,0.5569
7,add duration_exact_30,0.6713,0.7241,0.7172,0.5606
8,add duration_exact_60,0.6712,0.7241,0.7177,0.5581


Mean fold metrics, variability, and runtime


,config,accuracy,accuracy_sd,roc_auc,roc_auc_sd,success_f1,failure_recall,fit_seconds
0,original conservative,0.6693,0.0005,0.7240,0.0005,0.7158,0.5572,1.0108
1,without state/location,0.6616,0.0020,0.7106,0.0018,0.7098,0.5458,0.8791
2,without duration-derived,0.6707,0.0021,0.7240,0.0005,0.7170,0.5587,0.9488
3,without category-relative goal,0.6712,0.0020,0.7224,0.0008,0.7167,0.5625,0.8487
4,without redundant goal-derived,0.6712,0.0020,0.7224,0.0008,0.7167,0.5625,0.7833
5,add goal_vs_state,0.6720,0.0013,0.7244,0.0008,0.7184,0.5588,0.8356
6,add goal_round_1000,0.6711,0.0009,0.7239,0.0004,0.7178,0.5569,0.9145
7,add duration_exact_30,0.6713,0.0026,0.7241,0.0003,0.7172,0.5606,1.7435
8,add duration_exact_60,0.6712,0.0020,0.7242,0.0004,0.7177,0.5581,1.3845


Fold deltas versus original


,config,folds_accuracy_higher,fold_accuracy_delta,fold_auc_delta,fold_failure_recall_delta
0,without state/location,0,"[-0.0097, -0.0049, -0.0087]","[-0.0148, -0.0143, -0.0112]","[-0.0105, -0.0056, -0.0179]"
1,without duration-derived,2,"[0.0007, 0.0042, -0.0007]","[0.0, 0.0003, -0.0003]","[0.0005, 0.0027, 0.0012]"
2,without category-relative goal,2,"[0.0017, 0.0043, -0.0004]","[-0.0017, -0.0022, -0.001]","[0.0061, 0.0066, 0.0032]"
3,without redundant goal-derived,2,"[0.0017, 0.0043, -0.0004]","[-0.0017, -0.0022, -0.001]","[0.0061, 0.0066, 0.0032]"
4,add goal_vs_state,3,"[0.0037, 0.0019, 0.0024]","[-0.0001, -0.0, 0.0015]","[0.0015, 0.002, 0.0015]"
5,add goal_round_1000,3,"[0.0009, 0.0033, 0.001]","[-0.0002, -0.0001, -0.0001]","[0.0002, 0.0017, -0.0029]"
6,add duration_exact_30,2,"[-0.0013, 0.0047, 0.0024]","[-0.0005, 0.0003, 0.0006]","[0.0002, 0.0081, 0.002]"
7,add duration_exact_60,2,"[0.0, 0.0047, 0.0009]","[0.0001, 0.0005, 0.0]","[0.0002, 0.0032, -0.0007]"


## Interpretation

The comparisons are descriptive outer-fold development evidence, not a fresh independent estimate. A change is worth further consideration only if its pooled metrics improve without a meaningful trade-off, its fold deltas are directionally consistent, and its launch-time semantics are verified. No result here changes `us_primary_model_v1.joblib`.